# Skeleton-EAA-Pose Colab Pipeline

Module 1 filters PKU v1 daily actions. Module 2 is split into Step 2A Detect/Track and Step 2B Pose.


## 1. Mount Drive and Install Repo


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set this to your repo location in Colab.
!git pull origin main
!git clone https://github.com/tuan8p/Skeleton-EAA-Pose.git
REPO_PATH = '/content/Skeleton-EAA-Pose'
%cd {REPO_PATH}

# Base dependencies + YOLO tracking.
!pip install -q -r requirements.txt

# RTMW3D / MMPose stack for Step 2B.
!pip install -q -U openmim
!mim install -q mmengine mmcv mmdet mmpose


## 2. Dataset Paths


In [ ]:
# ?? Ch?n dataset ??????????????????????????????????????????????????????????????
DATASET = 'pku_v1'   # 'pku_v1' | 'pku_v2' | 'tsu'

DRIVE_ROOT = '/content/drive/MyDrive/ĐACN-TN_datasets/ĐATN/rawdatasets'

if DATASET == 'pku_v1':
    CONFIG_FILE  = f'{REPO_PATH}/configs/pku_v1.yaml'
    VIDEO_DIR    = f'{DRIVE_ROOT}/videos/PKUv1'
    SKELETON_DIR = f'{DRIVE_ROOT}/skeletons/PKU/Skeleton'
    LABEL_DIR    = f'{DRIVE_ROOT}/skeletons/PKU/Label_PKUMMD_v1'
    ACTIONS_XLSX = f'{DRIVE_ROOT}/skeletons/PKU/Actions.xlsx'

    FILTERED_LABEL_DIR    = f'{DRIVE_ROOT}/skeletons/PKU/Label_PKUMMDv1_daily'
    FILTERED_ACTIONS_CSV  = f'{DRIVE_ROOT}/skeletons/PKU/Actions_daily.csv'
    ACTIONS_FILE          = FILTERED_ACTIONS_CSV
    OUT_DIR               = f'{DRIVE_ROOT}/PKU_MMD_v1/samples_npy'

elif DATASET == 'pku_v2':
    CONFIG_FILE  = f'{REPO_PATH}/configs/pku_v2.yaml'
    VIDEO_DIR    = f'{DRIVE_ROOT}/videos/PKUv2'
    LABEL_DIR    = f'{DRIVE_ROOT}/skeletons/PKU/Label_PKUMMD_v2'
    ACTIONS_XLSX = f'{DRIVE_ROOT}/skeletons/PKU/Actions_v2.xlsx'

    FILTERED_LABEL_DIR    = LABEL_DIR
    ACTIONS_FILE          = ACTIONS_XLSX
    OUT_DIR               = f'{DRIVE_ROOT}/PKU_MMD_v2/samples_npy'

elif DATASET == 'tsu':
    CONFIG_FILE  = f'{REPO_PATH}/configs/tsu.yaml'
    VIDEO_DIR    = f'{DRIVE_ROOT}/videos/TSU'
    LABEL_DIR    = f'{DRIVE_ROOT}/skeletons/TSU/Annotation_v1.0'
    OUT_DIR      = f'{DRIVE_ROOT}/TSU_ske/samples_npy'

    FILTERED_LABEL_DIR    = LABEL_DIR
    ACTIONS_FILE          = None

DEVICE = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
print(f'Dataset: {DATASET}, Device: {DEVICE}')
print(f'Video dir: {VIDEO_DIR}')
print(f'Segments dir: {FILTERED_LABEL_DIR}')
print(f'Output dir: {OUT_DIR}')


## 2.1. Tracking Options


In [ ]:
# YOLO/Tracker options for Step 2A benchmarking.
TRACKING_MODEL = 'yolo26m.pt'       # e.g. 'yolo26n.pt' | 'yolo26s.pt' | 'yolo26m.pt'
TRACKING_TRACKER = 'bytetrack.yaml' # e.g. 'bytetrack.yaml' | 'botsort.yaml'
TRACK_START_INDEX = 0            # inclusive index in sorted full video list
TRACK_END_INDEX = None          # exclusive index; None means until the end
TRACK_QC_MAX_INTERP_GAP = 10    # max no_detection gap length for 2A_QC_2 bbox interpolation

# Run all pending videos in the selected index range by default.
# Set RUN_ALL_TRACKS=False and TRACK_LIMIT=N to process only N pending videos
# inside that range. Existing track JSON files are skipped after range filtering.
RUN_ALL_TRACKS = False
TRACK_LIMIT = 3

print(f'Tracking model: {TRACKING_MODEL}')
print(f'Tracker config: {TRACKING_TRACKER}')
print(f'Track index range: [{TRACK_START_INDEX}:{TRACK_END_INDEX}]')
print(f'Track QC max interpolation gap: {TRACK_QC_MAX_INTERP_GAP}')
print('Run all pending tracks:', RUN_ALL_TRACKS, 'Limit:', TRACK_LIMIT)


## 3. Module 1: Filter PKU v1 Daily Actions


In [ ]:
# if DATASET == 'pku_v1':
#     !python -m eaa_pose.filter_pku_interactions \
#         --config "{CONFIG_FILE}" \
#         --skeleton-dir "{SKELETON_DIR}" \
#         --label-dir "{LABEL_DIR}" \
#         --src-actions-xlsx "{ACTIONS_XLSX}" \
#         --out-label-dir "{FILTERED_LABEL_DIR}" \
#         --out-actions-csv "{FILTERED_ACTIONS_CSV}"
# else:
#     print('Module 1 filter skipped for', DATASET)


## 4. Step 2A: YOLO26 + ByteTrack Person Tracking


In [ ]:
track_scope_arg = '--all' if RUN_ALL_TRACKS else f'--limit {TRACK_LIMIT}'
track_range_arg = f'--start-index {TRACK_START_INDEX}'
if TRACK_END_INDEX is not None:
    track_range_arg += f' --end-index {TRACK_END_INDEX}'
cmd = f'''python -m eaa_pose.run_tracks \
    --config "{CONFIG_FILE}" \
    --video-dir "{VIDEO_DIR}" \
    --segments-dir "{FILTERED_LABEL_DIR}" \
    --actions-xlsx "{ACTIONS_FILE if ACTIONS_FILE else ''}" \
    --out-dir "{OUT_DIR}" \
    --device "{DEVICE}" \
    --tracking-model "{TRACKING_MODEL}" \
    --tracking-tracker "{TRACKING_TRACKER}" \
    {track_range_arg} \
    {track_scope_arg}'''
print(cmd)
!{cmd}


## 5. Inspect Track Stats


In [ ]:
import json
from pathlib import Path

track_stats_path = Path(OUT_DIR) / 'track_stats.json'
track_stats = json.loads(track_stats_path.read_text(encoding='utf-8'))
print(json.dumps(track_stats, indent=2, ensure_ascii=False)[:4000])


## 6. Step 2A_QC_1: Retry No-Detection Track Videos


In [ ]:
cmd = f'''python -m eaa_pose.run_track_qc_retry \
    --config "{CONFIG_FILE}" \
    --video-dir "{VIDEO_DIR}" \
    --segments-dir "{FILTERED_LABEL_DIR}" \
    --actions-xlsx "{ACTIONS_FILE if ACTIONS_FILE else ''}" \
    --out-dir "{OUT_DIR}" \
    --device "{DEVICE}" \
    --tracking-model "{TRACKING_MODEL}" \
    --tracking-tracker "{TRACKING_TRACKER}"'''
print(cmd)
!{cmd}


## 7. Step 2A_QC_2: Interpolate Short No-Detection Gaps


In [ ]:
cmd = f'''python -m eaa_pose.run_track_qc_interpolate \
    --config "{CONFIG_FILE}" \
    --out-dir "{OUT_DIR}" \
    --max-gap "{TRACK_QC_MAX_INTERP_GAP}"'''
print(cmd)
!{cmd}


## 8. Inspect Track QC Stats


In [ ]:
track_qc_stats_path = Path(OUT_DIR) / 'track_stats_qc.json'
if track_qc_stats_path.exists():
    with open(track_qc_stats_path, 'r', encoding='utf-8') as f:
        track_qc_stats = json.load(f)
    print(json.dumps(track_qc_stats.get('status_counts_in_action', {}), indent=2, ensure_ascii=False))
    for status, items in track_qc_stats.get('videos_by_status', {}).items():
        if items:
            print(status, len(items), items[:3])
else:
    print('Missing:', track_qc_stats_path)


## 9. Step 2B: RTMW3D Pose From Track JSON


In [ ]:
!python -m eaa_pose.run_pose \
    --config "{CONFIG_FILE}" \
    --video-dir "{VIDEO_DIR}" \
    --segments-dir "{FILTERED_LABEL_DIR}" \
    --actions-xlsx "{ACTIONS_FILE if ACTIONS_FILE else ''}" \
    --out-dir "{OUT_DIR}" \
    --device "{DEVICE}"


## 10. Inspect Metadata, Pose Stats, and QC Reports


In [ ]:
from pathlib import Path
import json

out_dir = Path(OUT_DIR)
for name in ['metadata.json', 'pose_stats.json']:
    p = out_dir / name
    print('\n====', name, '====')
    print(json.dumps(json.loads(p.read_text(encoding='utf-8')), indent=2, ensure_ascii=False)[:4000])

qc_files = sorted((out_dir / 'qc').glob('*_qc.json'))
print(f'QC reports: {len(qc_files)}')
if qc_files:
    print('First QC file:', qc_files[0])
    print(json.dumps(json.loads(qc_files[0].read_text(encoding='utf-8')), indent=2, ensure_ascii=False)[:4000])


## 11. Verify One SkateFormer Sample


In [ ]:
import glob
import numpy as np

npy_files = sorted(glob.glob(f'{OUT_DIR}/*.npy'))
print('Num samples:', len(npy_files))
if npy_files:
    arr = np.load(npy_files[0])
    print('Sample:', npy_files[0])
    print('Shape:', arr.shape)
    assert arr.ndim == 4
    assert arr.shape[0] == 3
    assert arr.shape[2] == 25
    assert arr.shape[3] == 1
    print('OK: expected SkateFormer layout (3, T, 25, 1)')
